In [8]:
%load_ext cuml.accel
%run /mnt/d/Users/Admin/Projects/dso/SAR_ML/notebooks/SSR/SSRtransforms_preloaded.py
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR
from torch.utils.data import DataLoader, ConcatDataset
import re
import copy
import torchvision
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from cuml.manifold import TSNE, UMAP
from joblib import Parallel, delayed
from tqdm import tqdm
import pickle

In [9]:
def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Uncomment only if you need 100% determinism and can handle errors
    # torch.use_deterministic_algorithms(True, warn_only=True)
    # os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    os.environ["PYTHONHASHSEED"] = str(seed)

def worker_init_fn(worker_id):
    """DataLoader worker init for reproducibility"""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [10]:
# workspace = "/workspace/alvin/SAR_ML"
workspace = "/mnt/d/Users/Admin/Projects/dso/SAR_ML"
data_workspace = os.path.join(workspace, "data/SAMPLE")
# clutter_dir = os.path.join(workspace, "data/MSTAR/CLUTTER/15_DEG")

In [11]:
with open(os.path.join(workspace, "weights/SSR/SAMPLE_synth_gmm_cache.pkl"), "rb") as f:
    gmm_cache = pickle.load(f)

In [12]:
synth_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/synth"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

SSR_synth_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/synth"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), SSRAugmentation(gmm_cache, alpha=0.6, beta=0.4, apply_prob=0.5, gaussian_noise = True, mu_s = 0.0, sigma_s = 0.3), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

meas_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/real"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

In [ ]:
device = torch.device("cuda")
model = models.resnet18(weights = None) # dont load ImageNet Weights
# new_model.fc = nn.Linear(new_model.fc.in_features, len(train_ds.class_to_idx))
model.fc = nn.Sequential(
    nn.Dropout(p = 0.4),
    nn.Linear(model.fc.in_features, len(synth_ds.class_to_idx))
)
# Load your trained weights
model.load_state_dict(torch.load(
    os.path.join(workspace, f"weights/SSR/Experiment_1/wo_aug/rn18_run0_b16.pth"),
    map_location=device
))

model = model.to(device)
model.eval()

feature_extractor = nn.Sequential(*list(model.children())[:-1])
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

features, labels_list = [], []

with torch.no_grad():
    for inputs, targets in DataLoader(
        synth_ds, batch_size = 16, 
        shuffle = False, num_workers = 8,
        pin_memory = True, persistent_workers = True):

        inputs = inputs.to(device)
        feats = feature_extractor(inputs)
        feats = feats.view(feats.size(0), -1)
        features.append(feats.cpu())
        labels_list.append(targets)

features = torch.cat(features).numpy()
labels = torch.cat(labels_list).numpy()


In [13]:
def _get_features_from_model(model, dataset, device):
    """
    parameters:
    model: PyTorch model to extract features from
    dataset: PyTorch Dataset to extract features for
    device: torch.device to perform computations on

    returns:
    features: cupy array of shape (num_samples, feature_dim) containing extracted features
    labels: cupy array of shape (num_samples,) containing corresponding labels
    """
    model = model.to(device)
    model.eval()

    feature_extractor = nn.Sequential(*list(model.children())[:-1])
    feature_extractor = feature_extractor.to(device)
    feature_extractor.eval()

    features, labels_list = [], []

    with torch.no_grad():
        for inputs, targets in DataLoader(
            dataset, batch_size = 16, 
            shuffle = False, num_workers = 8,
            pin_memory = True, persistent_workers = True):

            inputs = inputs.to(device)
            feats = feature_extractor(inputs)
            feats = feats.view(feats.size(0), -1)
            features.append(feats.cpu())
            labels_list.append(targets)

    features = cp.asarray(torch.cat(features).numpy())
    labels = cp.asarray(torch.cat(labels_list).numpy())

    return features, labels

In [14]:
def _gaussian_fit(features, labels, dataset):
    """
    Fit a Gaussian distribution to the features of a specific class.

    parameters:
    features: cupy array of shape (num_samples, feature_dim) containing extracted features
    labels: cupy array of shape (num_samples,) containing corresponding labels
    dataset: PyTorch Dataset containing the class information

    returns:
    class_mu: cupy array of shape (feature_dim,) containing the mean vector of the fitted Gaussian
    sigma_inv: cupy array of shape (feature_dim, feature_dim) containing the inverse covariance matrix of the fitted Gaussian
    """
    class_paras = {}
    for class_idx in dataset.class_to_idx.values():
        targets = (labels == class_idx)
        class_features = features[targets]

        class_mu = class_features.mean(axis = 0)
        class_paras[class_idx] = class_mu

    sigma = cp.cov(features, rowvar = False) # rowvar means observations are rows, features are columns
    sigma_inv = cp.linalg.inv(sigma)

    return class_paras, sigma_inv

In [15]:
def _mahalanobis_distance(feature, class_paras, sigma_inv):
    """
    Compute the Mahalanobis distance between a feature vector (usually test samples) and a class mean.

    parameters:
    feature: cupy array of shape (feature_dim,) containing the feature vector
    class_paras: dictionary mapping class indices to their mean vectors
    sigma_inv: cupy array of shape (feature_dim, feature_dim) containing the inverse covariance matrix

    returns:
    distance: vector representing the Mahalanobis distance from class distributions (shape of (num_classes,))
    """
    dist_vector = cp.zeros(len(class_paras))
    for class_idx, class_mu in class_paras.items():
        diff = feature - class_mu
        dist_vector[class_idx] = cp.sqrt(diff.T @ sigma_inv @ diff)
    return dist_vector

In [19]:
def _mahalanobis_distance_for_dataset(features, class_paras, sigma_inv):
    """
    Compute the Mahalanobis distance for a set of features against class distributions.

    parameters:
    features: cupy array of shape (num_samples, feature_dim) containing feature vectors
    class_paras: dictionary mapping class indices to their mean vectors
    sigma_inv: cupy array of shape (feature_dim, feature_dim) containing the inverse covariance matrix

    returns:
    distances: cupy array of shape (num_samples, num_classes) containing Mahalanobis distances to each class
    """
    num_samples = features.shape[0]
    num_classes = len(class_paras)
    distances = cp.zeros((num_samples, num_classes))

    for i in range(num_samples):
        distances[i, :] = _mahalanobis_distance(features[i], class_paras, sigma_inv)

    return distances

In [53]:
device = torch.device("cuda")
model = models.resnet18(weights = None) # dont load ImageNet Weights
# new_model.fc = nn.Linear(new_model.fc.in_features, len(train_ds.class_to_idx))
model.fc = nn.Sequential(
    nn.Dropout(p = 0.4),
    nn.Linear(model.fc.in_features, len(synth_ds.class_to_idx))
)
# Load your trained weights
model.load_state_dict(torch.load(
    os.path.join(workspace, f"weights/SSR/Experiment_1/SSR_exp1_full_runs/rn18_seed666_b16.pth"),
    map_location=device
))

# this gives the synth_features and synth_labels
synth_features, synth_labels = _get_features_from_model(model, SSR_synth_ds, device)
synth_class_paras, synth_sigma_inv = _gaussian_fit(synth_features, synth_labels, SSR_synth_ds)

meas_features, meas_labels = _get_features_from_model(model, meas_ds, device)
meas_distances = _mahalanobis_distance_for_dataset(meas_features, synth_class_paras, synth_sigma_inv)

In [54]:
pred_class = cp.argmin(meas_distances, axis = 1)

In [55]:
(pred_class == meas_labels).sum() / len(meas_labels)

array(0.88698885)